# Fisher-KPP Geo-Spectral Forward PINN Lab

This notebook runs the Colab-ready Fisher-KPP forward PINN experiment and writes the same diagnostics used by the repository scripts. The detailed method explanation, prior-work rationale, and observation analysis are maintained in `docs/fisher_kpp_pinn_review_response.docx`.

Use the configuration cell to choose the default Geo-Spectral forward profile, the simpler Korea pine-wilt style forward baseline, or the optional RK4-teacher-assisted variant.


In [ ]:
%matplotlib inline

from __future__ import annotations

import base64
import io
import shutil
import json
import sys
import zipfile
from dataclasses import replace
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import torch
from IPython.display import Image, Markdown, display

# Colab/self-contained bootstrap -------------------------------------------------
# If the local package is missing, this cell reconstructs the project files from an
# embedded source archive. That makes the notebook runnable after uploading only the
# .ipynb file to Google Colab.
_EMBEDDED_PROJECT_ZIP_B64 = """
...
"""

_EMBEDDED_PROJECT_VERSION = "ablowitz-zeppetella-wave"


def _find_project_root() -> Path | None:
    cwd = Path.cwd().resolve()
    for candidate in [cwd, *cwd.parents]:
        if (candidate / "fisher_origin_lab").exists():
            return candidate
    return None


def _running_in_colab() -> bool:
    try:
        import google.colab  # type: ignore  # noqa: F401
        return True
    except Exception:
        return False


def _bootstrap_embedded_project(*, refresh: bool = False) -> Path:
    target = Path("/content/fisher-kpp-origin-lab") if _running_in_colab() else Path.cwd().resolve() / "fisher-kpp-origin-lab"
    if refresh and target.exists():
        shutil.rmtree(target)
    target.mkdir(parents=True, exist_ok=True)
    raw = base64.b64decode("".join(_EMBEDDED_PROJECT_ZIP_B64.split()))
    with zipfile.ZipFile(io.BytesIO(raw)) as zf:
        zf.extractall(target)
    (target / ".embedded_project_version").write_text(_EMBEDDED_PROJECT_VERSION, encoding="utf-8")
    return target.resolve()

PROJECT_ROOT = _bootstrap_embedded_project(refresh=True) if _running_in_colab() else _find_project_root()
if PROJECT_ROOT is None:
    PROJECT_ROOT = _bootstrap_embedded_project(refresh=True)

if not (PROJECT_ROOT / "fisher_origin_lab").exists():
    raise RuntimeError(f"Could not locate or bootstrap fisher_origin_lab under {PROJECT_ROOT}")

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print(f"project root: {PROJECT_ROOT}")
print(f"torch: {torch.__version__} | cuda available: {torch.cuda.is_available()}")


## Plan

1. Select the forward profile and runtime size.
2. Preview truth fields and sensor locations.
3. Train the PINN and restore the best validation checkpoint.
4. Compare against the same-problem RK4 baseline and inspect reconstruction, front metrics, mass trajectory, PNG diagnostics, and GIF output.


In [ ]:
from fisher_origin_lab.config import (
    ExperimentConfig,
    LossWeights,
    ModelConfig,
    ObservationConfig,
)
from fisher_origin_lab.simulate import forward_fisher_kpp, sample_observations, truth_field_at
from fisher_origin_lab.train import run_experiment

# Notebook defaults now use the exact Ablowitz-Zeppetella traveling-wave benchmark.
USE_ABLOWITZ_ZEPPETELLA = True
USE_GEO_SPECTRAL_FORWARD = False
USE_KOREA_PINE_STYLE = False
USE_RK4_TEACHER_ASSIST = False
if USE_ABLOWITZ_ZEPPETELLA:
    RUN_NAME = "notebook_ablowitz_zeppetella"
else:
    RUN_NAME = "notebook_geo_spectral_forward" if USE_GEO_SPECTRAL_FORWARD else "notebook_korea_pine_style"
if USE_RK4_TEACHER_ASSIST:
    RUN_NAME = f"{RUN_NAME}_rk4_teacher"
QUICK = True
EPOCHS = 60
ENSEMBLE = 1
RUN_DIFFERENTIABLE_BASELINE = False
BASELINE_EPOCHS = 60
BASE_SEED = 7

# Paper-style front ablation knobs. Default keeps the stable analytic front-area constraint.
FRONT_AREA_WEIGHT = 1.0
FRONT_AREA_TEMPERATURE = 0.015
EXPECTED_FRONT_PDE_WEIGHT = 0.0
LEADING_EDGE_FLOOR_WEIGHT = 0.0

# Solver-assisted weak RK4 regularizer. Leave disabled for a pure PINN run.
RK4_TEACHER_WEIGHT = 0.005 if USE_RK4_TEACHER_ASSIST else 0.0
RK4_TEACHER_POOL = 4096 if USE_RK4_TEACHER_ASSIST else 0
RK4_TEACHER_BATCH = 512 if USE_RK4_TEACHER_ASSIST else 0
RK4_TEACHER_LATE_FRACTION = 0.0
RK4_PRETRAIN_STEPS = 0
RK4_PRETRAIN_BATCH = 0

base_cfg = ExperimentConfig(
    observations=ObservationConfig(samples_per_frame=500, noise_std=0.02, focus_fraction=0.5),
    model=ModelConfig(learn_drift=False, learn_diffusion=False, learn_reaction=False),
    weights=LossWeights(gradient=0.01),
    ensemble=ENSEMBLE,
    base_seed=BASE_SEED,
    out_dir=PROJECT_ROOT / "runs" / RUN_NAME,
    run_classical_baseline=RUN_DIFFERENTIABLE_BASELINE,
    baseline_epochs=BASELINE_EPOCHS,
)

if USE_ABLOWITZ_ZEPPETELLA:
    base_cfg = base_cfg.ablowitz_zeppetella_forward()
elif USE_GEO_SPECTRAL_FORWARD:
    base_cfg = base_cfg.geo_spectral_forward()
elif USE_KOREA_PINE_STYLE:
    base_cfg = base_cfg.korea_pine_style()

cfg = base_cfg.quick() if QUICK else base_cfg
cfg = replace(
    cfg,
    out_dir=PROJECT_ROOT / "runs" / RUN_NAME,
    ensemble=ENSEMBLE,
    run_classical_baseline=RUN_DIFFERENTIABLE_BASELINE,
    baseline_epochs=BASELINE_EPOCHS,
    weights=replace(
        cfg.weights,
        leading_edge_area=FRONT_AREA_WEIGHT,
        expected_front_pde=EXPECTED_FRONT_PDE_WEIGHT,
        leading_edge=LEADING_EDGE_FLOOR_WEIGHT,
        rk4_teacher=RK4_TEACHER_WEIGHT,
    ),
    train=replace(
        cfg.train,
        epochs=EPOCHS,
        print_every=max(1, EPOCHS // 4),
        leading_edge_area_temperature=FRONT_AREA_TEMPERATURE,
        rk4_teacher_pool=RK4_TEACHER_POOL,
        rk4_teacher_batch=RK4_TEACHER_BATCH,
        rk4_teacher_late_fraction=RK4_TEACHER_LATE_FRACTION,
        rk4_pretrain_steps=RK4_PRETRAIN_STEPS,
        rk4_pretrain_batch=RK4_PRETRAIN_BATCH,
    ),
)

print(json.dumps(cfg.to_dict(), indent=2, default=str))


## Data Preview

The synthetic observation design mixes uniform sensors with front-focused sensors. This avoids a degenerate dataset where most observations are nearly zero background.

In [ ]:
rng = np.random.default_rng(cfg.base_seed)
truth = forward_fisher_kpp(cfg.domain, cfg.pde, cfg.seed)
observations = sample_observations(truth, cfg.domain, cfg.observations, rng)

print(f"truth fields: {truth.fields.shape}")
print(f"observations: {observations.xyt.shape}, values: {observations.values.shape}")
print(f"value range: [{observations.values.min():.3f}, {observations.values.max():.3f}]")

times = [0.0, cfg.observations.start_time, cfg.domain.t_end]
fig, axes = plt.subplots(1, len(times), figsize=(12, 3.5), constrained_layout=True)
for ax, t in zip(axes, times):
    xs, field = truth_field_at(truth, t, n=96)
    ax.imshow(field.T, origin="lower", extent=[0, cfg.domain.box, 0, cfg.domain.box], cmap="magma", vmin=0, vmax=1)
    ax.plot(cfg.seed.center_x, cfg.seed.center_y, marker="*", color="cyan", markersize=12, markeredgecolor="white")
    if t == cfg.domain.t_end:
        latest = np.isclose(observations.xyt[:, 2], cfg.domain.t_end)
        ax.scatter(observations.xyt[latest, 0], observations.xyt[latest, 1], s=5, c="white", alpha=0.35, linewidths=0)
    ax.set_title(f"truth t={t:.2f}")
    ax.set_xticks([])
    ax.set_yticks([])
plt.show()

## Run The Forward PINN Experiment

This cell writes metrics and visual diagnostics to `cfg.out_dir`. The run exports observation coverage, reconstruction/error panels, space-time error trends, residual/front maps, RK4 comparison, adaptive loss multipliers, validation checkpoint diagnostics, and training diagnostics.


In [ ]:
metrics = run_experiment(cfg)
metrics_path = cfg.out_dir / "metrics.json"
figure_paths = [Path(path) for path in metrics.get("figures", [])]

print(f"metrics: {metrics_path}")
for path in figure_paths:
    print(f"figure : {path}")


## Metrics

For the Geo-Spectral forward profile, the primary checks are reconstruction error, train/validation observation MSE, RK4 same-problem accuracy, hard initial-condition residual, default PirateNet/RWF architecture setting, scaled TW features, optional NIF-Pirate ablation, optional weak RK4 teacher regularization, gradient-filtered moving-front speed loss, parabolic mass-balance loss, leading-edge front-area loss, learned `D/r`, boundary loss, front-local residual-gradient loss, residual-curriculum exponent, adaptive loss multipliers, and the restored best-validation epoch. Origin rows remain diagnostic because source-envelope inverse inference is intentionally off in this profile.


In [ ]:
def fmt_center(center):
    if center is None:
        return "-"
    return f"({center[0]:.3f}, {center[1]:.3f})"


def fmt_metric_value(value):
    if value is None:
        return "-"
    return f"{float(value):.4e}"

rows = []
for baseline in metrics["baselines"]:
    err = baseline["error"]
    rows.append([baseline["name"], fmt_center(baseline["center"]), "-" if err is None else f"{err:.4f}"])
method_name = "Geo-Spectral forward PINN" if cfg.model.use_geo_features else "Korea-style forward PINN"
rows.append([method_name, fmt_center(metrics["best_origin"]), f"{metrics['best_origin_error']:.4f}"])

md = "| method | center | origin diagnostic |\n|---|---:|---:|\n"
for name, center, err in rows:
    md += f"| {name} | {center} | {err} |\n"
display(Markdown(md))

front_rows = [
    ("final-time relative L2", metrics.get("final_time_relative_l2")),
    ("train observation MSE", metrics.get("train_observation_mse")),
    ("validation observation MSE", metrics.get("validation_observation_mse")),
    ("front area MAE, u>0.05", metrics.get("front_area_005_mae")),
    ("front area MAE, u>0.10", metrics.get("front_area_010_mae")),
    ("active-front band MAE", metrics.get("active_front_area_mae")),
    ("mass MAE", metrics.get("mass_mae")),
]
md = "| metric | value |\n|---|---:|\n"
for name, value in front_rows:
    md += f"| {name} | {fmt_metric_value(value)} |\n"
display(Markdown(md))

fg = metrics.get("front_geometry", {})
if fg:
    print("final active band truth/pinn:", round(fg["truth_active_band"][-1], 6), round(fg["pinn_active_band"][-1], 6))
    print("final area u>0.05 truth/pinn:", round(fg["area_above"]["0.05"]["truth"][-1], 6), round(fg["area_above"]["0.05"]["pinn"][-1], 6))
    print("final area u>0.10 truth/pinn:", round(fg["area_above"]["0.10"]["truth"][-1], 6), round(fg["area_above"]["0.10"]["pinn"][-1], 6))

print("learned physics:", {k: round(v, 6) for k, v in metrics["runs"][0]["physics"].items() if k in ["diffusion", "reaction", "velocity_x", "velocity_y"]})
print("geo:", cfg.geo)
print("front/mass weights:", {k: getattr(cfg.weights, k) for k in ["front_speed", "mass_balance", "leading_edge_area", "expected_front_pde", "leading_edge", "sparse"]})
print("adaptive/curriculum:", {"adaptive_loss_balancing": cfg.train.adaptive_loss_balancing, "residual_curriculum_epochs": cfg.train.residual_curriculum_epochs, "restore_best_validation": cfg.train.restore_best_validation, "residual_exponent": (cfg.train.residual_weight_exponent_start, cfg.train.residual_weight_exponent_end)})
print("model stabilizers:", {k: getattr(cfg.model, k) for k in ["fourier_sigma", "use_seed_front_features", "hard_initial_condition", "use_kpp_front_envelope", "front_envelope_margin", "front_envelope_width"]})


## PINN vs RK4 Accuracy

The RK4 baseline here is the RK4 time integrator adapted to the same 2D square-domain Fisher-KPP problem, Gaussian seed, and Neumann boundary condition as the PINN experiment. The Geo-Spectral PINN receives the same known Gaussian initial condition structurally, uses a KPP front-speed support envelope, adds an analytic leading-edge front-area constraint, and restores the best validation checkpoint before comparing with RK4. If `USE_RK4_TEACHER_ASSIST` is enabled, RK4 also supplies weak supervised pseudo-labels during training; report that run separately from the pure-PINN baseline.


In [ ]:
def display_accuracy_comparison(run_metrics, title="quick run"):
    rows = [
        ("PINN final relative L2 vs reference", run_metrics.get("pinn_final_time_relative_l2", run_metrics.get("final_time_relative_l2"))),
        ("RK4 final relative L2 vs reference", run_metrics.get("rk4_final_time_relative_l2")),
        ("PINN/RK4 final relative L2", run_metrics.get("pinn_vs_rk4_final_relative_l2")),
        ("PINN validation observation MSE", run_metrics.get("validation_observation_mse")),
        ("RK4 validation observation MSE", run_metrics.get("rk4_validation_observation_mse")),
        ("front area MAE, u>0.05", run_metrics.get("front_area_005_mae")),
        ("front area MAE, u>0.10", run_metrics.get("front_area_010_mae")),
        ("active-front band MAE", run_metrics.get("active_front_area_mae")),
        ("mass MAE", run_metrics.get("mass_mae")),
        ("RK4 runtime (sec)", run_metrics.get("rk4_runtime_sec")),
    ]
    md = f"### {title}\n\n| metric | value |\n|---|---:|\n"
    for name, value in rows:
        md += f"| {name} | {fmt_metric_value(value)} |\n"
    display(Markdown(md))

    comparison_path = next(
        (Path(path) for path in run_metrics.get("figures", []) if Path(path).name == "pinn_vs_rk4_comparison.png"),
        None,
    )
    if comparison_path is not None and comparison_path.exists():
        display(Image(filename=str(comparison_path)))


display_accuracy_comparison(metrics, title="quick run")


## Diagnostic Figures

The cells below display the generated observation, reconstruction, RK4 comparison, residual/front, training, and GIF diagnostics. Method details are kept in the DOCX report.


In [ ]:
for path in figure_paths:
    if path.name == "pinn_vs_rk4_comparison.png":
        continue
    if path.exists():
        display(Markdown(f"### {path.name}"))
        display(Image(filename=str(path)))


## Training Curves

The quick configuration is mainly a pipeline sanity check. The curves below expose known-IC loss, moving-front speed loss, parabolic mass-balance loss, leading-edge area loss, validation data loss, residual curriculum, and adaptive multipliers so unstable loss competition or overtraining is visible before running the full experiment.


In [ ]:
history = metrics["runs"][0]["history"]
epochs = [row["epoch"] for row in history]

fig, axes = plt.subplots(1, 3, figsize=(14, 3.6), constrained_layout=True)
axes[0].plot(epochs, [row["total"] for row in history], marker="o", label="total")
axes[0].plot(epochs, [row["data"] for row in history], marker="o", label="data")
axes[0].plot(epochs, [row["pde"] for row in history], marker="o", label="pde")
axes[0].plot(epochs, [row.get("ic", 0.0) for row in history], marker="o", label="known IC")
axes[0].plot(epochs, [row.get("mass", 0.0) for row in history], marker="o", label="mass balance")
axes[0].plot(epochs, [row.get("front_speed", 0.0) for row in history], marker="o", label="front speed")
axes[0].plot(epochs, [row.get("leading_edge_area", 0.0) for row in history], marker="o", label="front area")
if any(row.get("expected_front_pde", 0.0) > 0 for row in history):
    axes[0].plot(epochs, [row.get("expected_front_pde", 0.0) for row in history], marker="o", label="expected front PDE")
if any(row.get("leading_edge", 0.0) > 0 for row in history):
    axes[0].plot(epochs, [row.get("leading_edge", 0.0) for row in history], marker="o", label="leading-edge floor")
axes[0].set_yscale("log")
axes[0].set_xlabel("epoch")
axes[0].set_ylabel("loss")
axes[0].legend(fontsize=8)
axes[0].grid(True, alpha=0.25)

if cfg.model.use_source_envelope:
    axes[1].plot(epochs, [row["origin_error"] for row in history], marker="o", color="tab:green", label="origin error")
else:
    axes[1].plot(epochs, [row.get("bc", 0.0) for row in history], marker="o", label="bc")
    axes[1].plot(epochs, [row.get("front_grad", 0.0) for row in history], marker="o", label="front gPINN")
    axes[1].plot(epochs, [row.get("front_weight_mean", 1.0) for row in history], marker="o", label="front weight")
    axes[1].plot(epochs, [row.get("residual_exponent", 0.0) for row in history], marker="o", label="residual exponent")
    axes[1].plot(epochs, [row.get("sparse", 0.0) for row in history], marker="o", label="sparse L1")
axes[1].set_yscale("log")
axes[1].set_xlabel("epoch")
axes[1].set_ylabel("geo/front diagnostics")
axes[1].legend(fontsize=8)
axes[1].grid(True, alpha=0.25)

adaptive_keys = ["aw_data", "aw_pde", "aw_ic", "aw_bc", "aw_mass", "aw_front_grad", "aw_expected_front_pde", "aw_leading_edge"]
plotted = False
for key in adaptive_keys:
    if any(key in row for row in history):
        axes[2].plot(epochs, [row.get(key, np.nan) for row in history], marker="o", label=key.replace("aw_", ""))
        plotted = True
if plotted:
    axes[2].set_ylabel("adaptive multiplier")
    axes[2].legend(fontsize=8, ncol=2)
else:
    axes[2].plot(epochs, [row.get("elapsed_sec", 0.0) for row in history], marker="o", color="tab:gray")
    axes[2].set_ylabel("elapsed sec")
axes[2].set_xlabel("epoch")
axes[2].grid(True, alpha=0.25)
plt.show()


## Optional Full Run

Set `RUN_FULL = True` for a stronger experiment. The full run writes the same diagnostic figure set, including moving-front, leading-edge front-area, mass-balance, and RK4 comparison visuals, so smoke, quick, and full settings can be compared with the same metrics. If `USE_RK4_TEACHER_ASSIST` is enabled, the full run keeps the same weak solver-assisted loss active.


In [ ]:
RUN_FULL = False

if RUN_FULL:
    full_weights = replace(
        base_cfg.weights,
        leading_edge_area=FRONT_AREA_WEIGHT,
        expected_front_pde=EXPECTED_FRONT_PDE_WEIGHT,
        leading_edge=LEADING_EDGE_FLOOR_WEIGHT,
        rk4_teacher=RK4_TEACHER_WEIGHT,
    )
    full_train = replace(
        base_cfg.train,
        epochs=1200,
        print_every=100,
        rk4_teacher_pool=RK4_TEACHER_POOL,
        rk4_teacher_batch=RK4_TEACHER_BATCH,
        rk4_teacher_late_fraction=RK4_TEACHER_LATE_FRACTION,
        rk4_pretrain_steps=RK4_PRETRAIN_STEPS,
        rk4_pretrain_batch=RK4_PRETRAIN_BATCH,
    )
    full_cfg = replace(
        base_cfg,
        out_dir=PROJECT_ROOT / "runs" / ("notebook_geo_spectral_full" if USE_GEO_SPECTRAL_FORWARD else "notebook_forward_full"),
        ensemble=1,
        run_classical_baseline=True,
        baseline_epochs=250,
        weights=full_weights,
        train=full_train,
    )
    full_metrics = run_experiment(full_cfg)
    full_figure_paths = [Path(path) for path in full_metrics.get("figures", [])]
    display_accuracy_comparison(full_metrics, title="full run")
    for path in full_figure_paths:
        if path.name == "pinn_vs_rk4_comparison.png":
            continue
        if path.exists():
            display(Markdown(f"### full run: {path.name}"))
            display(Image(filename=str(path)))
else:
    print("RUN_FULL is False. Flip it to True when you want the slower validation run.")


## Optional Forward Ablation Matrix

Run this after the quick experiment when you want a paper-style method comparison for the forward Fisher-KPP setting. The default here is a very small smoke matrix; switch to `--preset quick --seeds 7,8,9` for a more useful comparison. The matrix includes the weak RK4 teacher, NIF/PirateNet variants, and the `geo_levelset_time_slab` moving-front geometry ablation. The summary plot reports final L2, `u>0.10` front-area MAE, and mass MAE, not inverse-origin error.


In [ ]:
RUN_ABLATION = False

if RUN_ABLATION:
    import subprocess

    cmd = [
        sys.executable,
        str(PROJECT_ROOT / "scripts" / "run_forward_ablation.py"),
        "--preset", "smoke",
        "--seeds", "7",
        "--out-dir", str(PROJECT_ROOT / "runs" / "notebook_forward_ablation_smoke"),
    ]
    subprocess.run(cmd, check=True)
    display(Image(filename=str(PROJECT_ROOT / "runs" / "notebook_forward_ablation_smoke" / "summary.png")))
else:
    print("RUN_ABLATION is False. Flip it to True for the optional forward ablation smoke matrix.")


## Notes For Reporting

- Use the DOCX report for method explanation and literature rationale.
- Report `validation_observation_mse`; training-only data fit is not enough.
- Report `pinn_final_time_relative_l2`, `rk4_final_time_relative_l2`, and `pinn_vs_rk4_final_relative_l2` together.
- Report `front_area_005_mae`, `front_area_010_mae`, `active_front_area_mae`, and `mass_mae`; final-time L2 alone misses moving-front failures.
- Treat quick Colab runs as diagnostics unless the metrics and generated figures support a stronger claim.


## Optional Long-Time rho(t) Curve PINN

This optional section targets only the right-panel-style damped `rho(t)` trend. It is intentionally separate from the Fisher-KPP front experiment above: the scalar curve is modeled as a damped oscillator ODE and solved with a small ODE-PINN under the same fair long-time parameters used by the numerical-integrator comparison.


In [ ]:
RUN_CURVE_PINN = True

if RUN_CURVE_PINN:
    from fisher_origin_lab.curve_trend import (
        CurvePINNConfig,
        CurveTrendConfig,
        integrate_curve,
        save_curve_pinn_outputs,
        train_curve_pinn,
    )

    curve_out_dir = PROJECT_ROOT / "runs" / "notebook_long_time_curve_pinn"
    curve_device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    trend_cfg = CurveTrendConfig()
    curve_pinn_cfg = CurvePINNConfig().quick()
    curve_baselines = {
        method: integrate_curve(method, trend_cfg)
        for method in ("forward_euler", "backward_euler", "trapezoidal", "rk4")
    }
    curve_result = train_curve_pinn(trend_cfg, curve_pinn_cfg, device=curve_device, seed=cfg.base_seed)
    curve_outputs = save_curve_pinn_outputs(curve_out_dir, curve_result, curve_baselines)

    rows = [
        ["PINN", curve_result["metrics"]["max_abs_error"], curve_result["metrics"]["relative_l2_to_exact"], curve_result["metrics"]["final_rho"]]
    ]
    for method, result in curve_baselines.items():
        rel_l2 = np.linalg.norm(result["rho"] - result["exact_rho"]) / (np.linalg.norm(result["exact_rho"]) + 1.0e-12)
        rows.append([method, float(result["abs_error"].max()), float(rel_l2), float(result["rho"][-1])])

    md = "| method | max abs error | L2 vs exact | final rho |\n|---|---:|---:|---:|\n"
    for name, max_err, rel_l2, final_rho in rows:
        md += f"| {name} | {max_err:.3e} | {rel_l2:.3e} | {final_rho:.4f} |\n"
    display(Markdown(md))
    display(Image(filename=curve_outputs["curve_png"]))
    display(Image(filename=curve_outputs["diagnostics_png"]))
    print("curve outputs:", curve_outputs)
